In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# LoRA 配置
lora_config = LoraConfig(
    r=16,  # LoRA 秩，可以根据效果调整
    lora_alpha=32, # LoRA 缩放因子
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # 目标模块，通常是注意力层的投影矩阵
    bias="none", # 是否对 bias 进行 LoRA
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

# 准备模型进行 LoRA 训练（例如，量化后训练）
# 如果你的GPU内存不足，可以尝试加载4bit量化的模型
# from transformers import BitsAndBytesConfig
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )
# model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
# model = prepare_model_for_kbit_training(model) # 准备4bit模型进行训练

# 应用 LoRA 配置
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # 打印可训练参数数量

# Tokenizer 的 pad_token_id 设置
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Qwen 默认的padding_side是right

# 数据整理器
# DataCollatorForLanguageModeling 会将文本分词并填充到相同的长度，并创建标签
# mlm=False 表示我们进行因果语言建模（Causal Language Modeling），即预测下一个词
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 准备训练数据
def preprocess_function(examples):
    # 将对话历史转换为模型期望的格式
    # Qwen 的 chat_template 会将 messages 转换为类似 chatml 的格式
    processed_examples = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        processed_examples.append(text)
    return tokenizer(processed_examples, truncation=True, max_length=512) # 截断和填充

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["messages"]
)

print(tokenized_dataset)
print(tokenized_dataset["train"][0])


# 训练参数
training_args = TrainingArguments(
    output_dir="./qwen_finetuned_tea", # 输出目录
    num_train_epochs=3, # 训练轮数，可以根据数据集大小调整
    per_device_train_batch_size=2, # 每个设备上的训练批次大小
    gradient_accumulation_steps=4, # 梯度累积步数，增加等效批次大小
    gradient_checkpointing=True, # 启用梯度检查点，节省内存
    optim="paged_adamw_8bit", # 优化器
    learning_rate=2e-4, # 学习率
    fp16=True, # 启用混合精度训练
    save_strategy="epoch", # 每个 epoch 保存一次模型
    logging_steps=10, # 日志记录步数
    report_to="none", # 不报告到外部服务
    # push_to_hub=True, # 如果想推送到 Hugging Face Hub
    # hub_model_id="your-hf-username/qwen-finetuned-tea",
)

# 初始化 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)